In [1]:
from skywork_o1_prm_inference.prm_model import PRM_MODEL
from transformers import AutoModel

/cpfs02/user/liurunze/miniforge3/envs/zj_GenPRM/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/cpfs02/user/liurunze/miniforge3/envs/zj_GenPRM/lib/python3.10/site-packages/_distutils_hack/__init__.py:54: UserWarning: Reliance on distutils from stdlib is deprecated. Users must rely on setuptools to provide the distutils module. Avoid importing distutils or import setuptools first, and avoid setting SETUPTOOLS_USE_DISTUTILS=stdlib. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


In [2]:
model_origin = AutoModel.from_pretrained("/mnt/workspace/hf_models/models--Skywork--Skywork-o1-Open-PRM-Qwen-2.5-7B")

In [4]:
for name, param in model_origin.named_parameters():
    print(f"{name:<60} {param[0,:10]}")
    break

pretrained_model.embed_tokens.weight                         tensor([ 0.0071, -0.0161, -0.0114, -0.0092,  0.0131, -0.0053,  0.0118,  0.0184,
        -0.0270, -0.0315], grad_fn=<SliceBackward0>)


In [2]:
model_origin = PRM_MODEL.from_pretrained("/mnt/workspace/hf_models/models--Skywork--Skywork-o1-Open-PRM-Qwen-2.5-7B")

/mnt/workspace/_/simpleRL-reason/verl/workers/critic/skywork_o1_prm_inference/modeling_base.py:344: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = loading_func(f

In [5]:
for name, param in model_origin.named_parameters():
    if "v_head" in name:
        print(f"{name:<60} {param[0,:10]}")
        break

v_head.summary.weight                                        tensor([ 0.0086, -0.0074, -0.0031,  0.0076, -0.0157,  0.0100, -0.0034,  0.0084,
         0.0024, -0.0021], grad_fn=<SliceBackward0>)


In [25]:
model_origin.pretrained_model.save_pretrained("/mnt/workspace/_/simpleRL-reason/_outputs/checkpoints/tmp", max_shard_size="999GB",
    safe_serialization=False  # This is the key argument!
)

In [29]:
model_tmp = PRM_MODEL.from_pretrained("/mnt/workspace/_/simpleRL-reason/_outputs/checkpoints/tmp1", strict=False)

TypeError: Qwen2Model.__init__() got an unexpected keyword argument 'strict'

In [46]:
import torch
from collections import OrderedDict
from transformers import Qwen2Model, Qwen2Config

config_path = "/mnt/workspace/_/simpleRL-reason/_outputs/checkpoints/tmp1"

config = Qwen2Config.from_pretrained(config_path)
original_model = Qwen2Model(config)

In [36]:
original_state_dict = original_model.state_dict()
prefixed_state_dict = OrderedDict()
prefix_to_add = "pretrained_model." # This is the prefix causing your problem

# Add the prefix to every key
for key, value in original_state_dict.items():
    prefixed_state_dict[prefix_to_add + key] = value

In [28]:
for name, param in model_origin.named_parameters():
    if "v_head" in name:
        print(f"{name:<60} {param[0,:10]}")
        break

v_head.summary.weight                                        tensor([ 0.0086, -0.0074, -0.0031,  0.0076, -0.0157,  0.0100, -0.0034,  0.0084,
         0.0024, -0.0021], grad_fn=<SliceBackward0>)


In [39]:
bin_file_path = config_path + "/pytorch_model.bin"
state_dict = torch.load(bin_file_path, map_location="cpu")

/tmp/ipykernel_2853/2685731235.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(bin_file_path, map_location="cpu")


In [51]:
new_state_dict = OrderedDict()
for name, param in state_dict.items():
    if name.startswith("pretrained_model."):
        new_state_dict[name.removeprefix("pretrained_model.")] = param

In [54]:
original_model.load_state_dict(new_state_dict)

In [59]:
for name, param in original_model.named_parameters():
    # if "v_head" in name:
        print(f"{name:<60} {param[0,:10]}")
        break

embed_tokens.weight                                          tensor([ 0.0071, -0.0161, -0.0114, -0.0092,  0.0131, -0.0053,  0.0118,  0.0184,
        -0.0270, -0.0315], grad_fn=<SliceBackward0>)


In [ ]:
original_model

In [22]:
model_load = AutoModel.from_pretrained("/mnt/workspace/_/simpleRL-reason/_outputs/checkpoints/verl-ppoFREEZE__models--Qwen--Qwen2.5-7B_models--Skywork--Skywork-o1-Open-PRM-Qwen-2.5-7B_simplelr_qwen_level3to5_max_response8192_batch1024_rollout8_klcoef0.0001_entcoef0.001/global_step_95/critic/huggingface")

Loading checkpoint shards: 100%|██████████| 6/6 [00:03<00:00,  1.55it/s]
Some weights of the model checkpoint at /mnt/workspace/_/simpleRL-reason/_outputs/checkpoints/verl-ppoFREEZE__models--Qwen--Qwen2.5-7B_models--Skywork--Skywork-o1-Open-PRM-Qwen-2.5-7B_simplelr_qwen_level3to5_max_response8192_batch1024_rollout8_klcoef0.0001_entcoef0.001/global_step_95/critic/huggingface were not used when initializing Qwen2Model: {'pretrained_model.layers.10.input_layernorm.weight', 'pretrained_model.layers.7.post_attention_layernorm.weight', 'pretrained_model.layers.26.self_attn.o_proj.weight', 'pretrained_model.layers.15.self_attn.k_proj.bias', 'pretrained_model.layers.1.mlp.up_proj.weight', 'pretrained_model.layers.13.mlp.down_proj.weight', 'pretrained_model.layers.3.self_attn.k_proj.bias', 'pretrained_model.layers.19.self_attn.k_proj.bias', 'pretrained_model.layers.9.self_attn.k_proj.bias', 'pretrained_model.layers.14.input_layernorm.weight', 'pretrained_model.layers.4.self_attn.k_proj.weight',

In [35]:
model_load1 = AutoModel.from_pretrained("/mnt/workspace/_/simpleRL-reason/_outputs/checkpoints/verl-ppoFREEZE__models--Qwen--Qwen2.5-7B_models--Skywork--Skywork-o1-Open-PRM-Qwen-2.5-7B_simplelr_qwen_level3to5_max_response8192_batch1024_rollout8_klcoef0.0001_entcoef0.001/global_step_5/critic/huggingface")

Loading checkpoint shards: 100%|██████████| 6/6 [00:04<00:00,  1.48it/s]
Some weights of the model checkpoint at /mnt/workspace/_/simpleRL-reason/_outputs/checkpoints/verl-ppoFREEZE__models--Qwen--Qwen2.5-7B_models--Skywork--Skywork-o1-Open-PRM-Qwen-2.5-7B_simplelr_qwen_level3to5_max_response8192_batch1024_rollout8_klcoef0.0001_entcoef0.001/global_step_5/critic/huggingface were not used when initializing Qwen2Model: {'pretrained_model.layers.10.input_layernorm.weight', 'pretrained_model.layers.7.post_attention_layernorm.weight', 'pretrained_model.layers.26.self_attn.o_proj.weight', 'pretrained_model.layers.15.self_attn.k_proj.bias', 'pretrained_model.layers.1.mlp.up_proj.weight', 'pretrained_model.layers.13.mlp.down_proj.weight', 'pretrained_model.layers.3.self_attn.k_proj.bias', 'pretrained_model.layers.19.self_attn.k_proj.bias', 'pretrained_model.layers.9.self_attn.k_proj.bias', 'pretrained_model.layers.14.input_layernorm.weight', 'pretrained_model.layers.4.self_attn.k_proj.weight', 

In [34]:
for name, param in model1.named_parameters():
    print(f"{name:<60} {param[0,:10]}")
    break

for name, param in model_load.named_parameters():
    print(f"{name:<60} {param[0,:10]}")
    break
    


pretrained_model.embed_tokens.weight                         tensor([ 0.0071, -0.0161, -0.0114, -0.0092,  0.0131, -0.0053,  0.0118,  0.0184,
        -0.0270, -0.0315], grad_fn=<SliceBackward0>)
embed_tokens.weight                                          tensor([-0.0071, -0.0193,  0.0044,  0.0029, -0.0404,  0.0031,  0.0139,  0.0183,
         0.0135, -0.0052], grad_fn=<SliceBackward0>)


In [36]:
for name, param in model_load.named_parameters():
    print(f"{name:<60} {param[0,:10]}")
    break

embed_tokens.weight                                          tensor([-0.0211, -0.0086,  0.0110,  0.0300,  0.0172,  0.0288,  0.0092,  0.0075,
        -0.0252, -0.0045], grad_fn=<SliceBackward0>)


In [16]:
_.save_pretrained(
    "/mnt/workspace/_/simpleRL-reason/_outputs/checkpoints/verl-ppoFREEZE__models--Qwen--Qwen2.5-7B_models--Skywork--Skywork-o1-Open-PRM-Qwen-2.5-7B_simplelr_qwen_level3to5_max_response8192_batch1024_rollout8_klcoef0.0001_entcoef0.001/global_step_95/critic/tmp",
    max_shard_size="999GB",
    safe_serialization=False  # This is the key argument!
)

In [18]:
model = PRM_MODEL.from_pretrained("/mnt/workspace/_/simpleRL-reason/_outputs/checkpoints/verl-ppoFREEZE__models--Qwen--Qwen2.5-7B_models--Skywork--Skywork-o1-Open-PRM-Qwen-2.5-7B_simplelr_qwen_level3to5_max_response8192_batch1024_rollout8_klcoef0.0001_entcoef0.001/global_step_95/critic/model")

In [37]:
model.to("cuda")

PRM_MODEL(
  (pretrained_model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584, padding_idx=151643)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=

In [20]:
model1 = PRM_MODEL.from_pretrained("/mnt/workspace/hf_models/models--Skywork--Skywork-o1-Open-PRM-Qwen-2.5-7B")

In [21]:
model1

PRM_MODEL(
  (pretrained_model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584, padding_idx=151643)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=

In [13]:
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
import os
os.environ["RANK"] = "0"
os.environ["WORLD_SIZE"] = "1"
os.environ["MASTER_ADDR"] = "1"
os.environ["MASTER_PORT"] = "1"
import torch
torch.distributed.init_process_group(backend="nccl")
fs_model = FSDP(model)

[W627 09:07:13.923074381 socket.cpp:697] [c10d] The client socket has failed to connect to [::ffff:0.0.0.1]:1 (errno: 110 - Connection timed out).


KeyboardInterrupt: 

In [3]:
model.state_dict()

NotImplementedError: 

In [5]:
import torch
# model.train()
model_origin.eval()
print(model_origin(torch.tensor([[0, 1111, 2222, 3333]])))
print(model_origin(torch.tensor([[0, 1111, 2222, 3333]]), return_probs=True))

(None, None, tensor([[ 2.0818, -1.2967, -0.6359, -0.9157]], grad_fn=<SqueezeBackward1>))
(None, None, tensor([[0.8891, 0.2147, 0.3462, 0.2858]], grad_fn=<SigmoidBackward0>))


In [8]:
from transformers import AutoModelForTokenClassification
from transformers import AutoConfig

critic_model_config = AutoConfig.from_pretrained('/mnt/workspace/hf_models/models--Skywork--Skywork-o1-Open-PRM-Qwen-2.5-1.5B', trust_remote_code=False)
critic_model_config.num_labels = 1

model2= AutoModelForTokenClassification.from_pretrained('/mnt/workspace/hf_models/models--Skywork--Skywork-o1-Open-PRM-Qwen-2.5-1.5B',
                                                       config=critic_model_config,)

Some weights of Qwen2ForTokenClassification were not initialized from the model checkpoint at /mnt/workspace/hf_models/models--Skywork--Skywork-o1-Open-PRM-Qwen-2.5-1.5B and are newly initialized: ['score.bias', 'score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
import torch
model2.train()
model2(torch.tensor([[0, 1111, 2222, 3333]]))

TokenClassifierOutput(loss=None, logits=tensor([[[-2.6802],
         [ 1.8139],
         [-0.7299],
         [-0.5687]]], grad_fn=<ViewBackward0>), hidden_states=None, attentions=None)

In [26]:
import torch.nn as nn

def find_dropout_layers(module, path=""):
    found = False
    for name, submodule in module.named_children():
        new_path = f"{path}.{name}" if path else name
        if isinstance(submodule, nn.Dropout):
            print(f"Found Dropout Layer at: {new_path} with p={submodule.p}")
            found = True
        else:
            found = find_dropout_layers(submodule, new_path) or found
    return found

print("Searching for nn.Dropout layers...")
was_found = find_dropout_layers(model)
if not was_found:
    print("No nn.Dropout layers were found. Dropout is likely applied functionally (F.dropout).")

Searching for nn.Dropout layers...
Found Dropout Layer at: v_head.dropout with p=0.1


In [ ]:
model

In [ ]:
---

In [ ]:
import torch
from collections import OrderedDict
from transformers import Qwen2Model, Qwen2Config

config_path = "/mnt/workspace/_/simpleRL-reason/_outputs/checkpoints/tmp1"

config = Qwen2Config.from_pretrained(config_path)
original_model = Qwen2Model(config)